In [1]:
cd /work/FAC/FBM/DBC/cdessim2/default/dmoi/projects/aars/aarsonline.github.io

/work/FAC/FBM/DBC/cdessim2/default/dmoi/projects/aars/aarsonline.github.io


In [2]:
datadir = '/work/FAC/FBM/DBC/cdessim2/default/dmoi/projects/aars/aarsonline.github.io/struct_treesmk2/'


In [3]:
#run pdbfixer on all structs
from Bio import PDB
from pdbfixer import PDBFixer
from openmm.app import PDBFile
import glob
import sys
import tqdm
import pandas as pd
import multiprocessing as mp
#argument parser for the script 
from concurrent.futures import TimeoutError
from pebble import ProcessPool, ProcessExpired
import os
import glob

In [4]:
#create a dataframe with all the pdb files
maindir = '/work/FAC/FBM/DBC/cdessim2/default/dmoi/projects/aars/aarsonline.github.io/'
c1 = glob.glob(maindir + 'class1/*/*/structures/*.pdb')
c2 = glob.glob(maindir + 'class2/*/*/structures/*.pdb')

domains = glob.glob(maindir + '*/*/data/domains/*/structures/*.pdb')

all_pdbs = c1 + c2 + domains
#remove duplicates
all_pdbs = list(set(all_pdbs))

pdbs = { i:{ 'class': pdb.split('/')[11], 'res': pdb.split('/')[12] , 'path': pdb } for i,pdb in enumerate(all_pdbs) }
print(f'Found {len(pdbs)} pdb files')

Found 3371 pdb files


In [5]:

pdb_df = pd.DataFrame.from_dict(pdbs, orient='index')
pdb_df['pdb_id'] = pdb_df['path'].apply(lambda x: os.path.basename(x).split('.')[0])
pdb_df['domain'] = pdb_df['path'].apply(lambda x: x.split('/')[15] if 'domains' in x else None)
print(pdb_df.head())

    class   res                                               path  \
0  class2  phe4  /work/FAC/FBM/DBC/cdessim2/default/dmoi/projec...   
1  class2   sep  /work/FAC/FBM/DBC/cdessim2/default/dmoi/projec...   
2  class1   trp  /work/FAC/FBM/DBC/cdessim2/default/dmoi/projec...   
3  class2   thr  /work/FAC/FBM/DBC/cdessim2/default/dmoi/projec...   
4  class2   thr  /work/FAC/FBM/DBC/cdessim2/default/dmoi/projec...   

                                              pdb_id            domain  
0   PheRS-Aβ_AF_Euk_Astyanax_mexicanus_gene103044481         Protozyme  
1                  SepRS_PDB_Arch_2ODR_M_maripadulis              None  
2                    TrpRS_PDB_Euk_Homo_sapiens_1R6T  Catalytic_domain  
3  ThrRS_AF_Euk_Saccharomyces_cerevisiae_S288C_ge...                N1  
4  ThrRS_AF_Mito_Oryza_sativa_Japonica_Group_gene...          ThrRS_IM  


In [6]:
#print the number of pdb files per class
print(pdb_df['class'].value_counts())

class
class1    1750
class2    1621
Name: count, dtype: int64


In [7]:
print( pdb_df['domain'].value_counts() )

domain
Protozyme                        548
Catalytic_domain                 548
CP2                               99
Anticodon_binding_domain_1a       99
Anticodon_binding_domain_HGPT     97
                                ... 
Lysine-rich                        3
Beta_chain                         3
Ergosterol_binding_domain          1
GluRS                              1
ProRS                              1
Name: count, Length: 64, dtype: int64


In [8]:
#drop all except Protozyme and Catalytic domains
keep = ['Protozyme', 'Catalytic_domain']
pdb_df = pdb_df[pdb_df['domain'].isin(keep)]
print(f'Keeping {len(pdb_df)} pdb files after filtering for domains {keep}')

Keeping 1096 pdb files after filtering for domains ['Protozyme', 'Catalytic_domain']


In [9]:
#print the number of pdb files per class / residue
print(pdb_df.groupby(['class', 'res']).size())

class   res 
class1  arg     48
        cys     46
        gln     30
        glu1    38
        glu2    10
        glu3    14
        ile     48
        leu1    32
        leu2    22
        lys     18
        met     50
        trp     50
        tyr     50
        val     46
class2  ala     40
        asn     44
        asp1    28
        asp2    30
        gly1    16
        gly2    24
        gly3    12
        his     46
        lys     44
        phe1    20
        phe2    24
        phe3    20
        phe4    18
        phe5    14
        pro1    32
        pro2    40
        pyl     18
        sep     12
        ser1    54
        ser2    10
        thr     48
dtype: int64


In [10]:
structs_all = glob.glob(datadir+'*/structures/*.pdb')
print( len(structs_all))

0


In [11]:
#how many fams are there? group by class and residue and domain
print(pdb_df.groupby(['class', 'res', 'domain']).size())
#are there any empty classes?
print(pdb_df[pdb_df['class'].isnull()])

class   res   domain          
class1  arg   Catalytic_domain    24
              Protozyme           24
        cys   Catalytic_domain    23
              Protozyme           23
        gln   Catalytic_domain    15
                                  ..
class2  ser1  Protozyme           27
        ser2  Catalytic_domain     5
              Protozyme            5
        thr   Catalytic_domain    24
              Protozyme           24
Length: 70, dtype: int64
Empty DataFrame
Columns: [class, res, path, pdb_id, domain]
Index: []


In [29]:
#make a directory for the results
if not os.path.exists(datadir):
	os.makedirs(datadir )

#make a folder for each class, residue and domain
pdb_df['struct_dir'] = pdb_df.apply(lambda x: os.path.join(datadir, x['class'], x['res'], x['domain']), axis=1)
for struct_dir in pdb_df['struct_dir'].unique():
	if not os.path.exists(struct_dir):
		os.makedirs(struct_dir)
		if not os.path.exists(os.path.join(struct_dir, 'structs')):
			os.makedirs(os.path.join(struct_dir, 'structs'))
		if not os.path.exists(os.path.join(struct_dir, 'pdbfixer_in')):
			os.makedirs(os.path.join(struct_dir, 'pdbfixer_in'))

	#create a dummy identifiers.txt file
	with open(os.path.join(struct_dir, 'identifiers.txt'), 'w') as f:
		f.write('This is a dummy file to indicate that this directory has been processed by pdbfixer.\n')
		f.write('You can remove this file if you want to reprocess the directory.\n')


In [13]:
#print the tree structure of the directories
def print_tree(path, level=0):
	if os.path.isdir(path):
		print(' ' * level + os.path.basename(path) + '/')
		for item in os.listdir(path):
			print_tree(os.path.join(path, item), level + 2)
	else:
		print(' ' * level + os.path.basename(path))

print_tree(datadir)

/
  class1/
    cys/
      Catalytic_domain/
        pdbfixer_in/
          CysRS_AF_Arch_Aciduliprofundum_boonei_T469_gene8827372.pdb
          CysRS_AF_Euk_Homo_sapiens_gene833.pdb
          CysRS_AF_Mito_Morone_saxatilis_gene118339815.pdb
          CysRS_AF_Arch_Candidatus_Nitrosopumilus_sediminis_gene13696907.pdb
          CysRS_AF_Euk_Arabidopsis_thaliana_gene833874.pdb
          CysRS_AF_Bact_Lactobacillus_amylovorus_GRL1118_gene66523130.pdb
          CysRS_AF_Bact_Prochlorococcus_marinus_subsp_marinus_str_CCMP1375_gene54200571.pdb
          CysRS_AF_Euk_Micromonas_commoda_gene8246535.pdb
          CysRS_AF_Bact_Treponema_pallidum_subsp_pertenue_str_SamoaD_gene57878631.pdb
          CysRS_AF_Euk_Drosophila_melanogaster_gene36784.pdb
          CysRS_AF_Mito_Saccharomyces_cerevisiae_S288C_gene855474.pdb
          CysRS_PDB_Bact_M_smegmatis_3C8Z.pdb
          CysRS_AF_Arch_Pyrobaculum_ferrireducens_gene11593896.pdb
          CysRS_AF_Arch_Methanospirillum_hungatei_JF-1_gene3922979.p

In [14]:
#copy all pdb files to the pdbfixer_in directory
for i, row in tqdm.tqdm(pdb_df.iterrows() , total=len(pdb_df), desc='Copying PDB files to pdbfixer_in'):
	src = row['path']
	dest = os.path.join(row['struct_dir'], 'pdbfixer_in', os.path.basename(src))
	if not os.path.exists(dest):
		os.makedirs(os.path.dirname(dest), exist_ok=True)
		os.system(f'cp {src} {dest}')

Copying PDB files to pdbfixer_in: 100%|██████████| 1096/1096 [00:03<00:00, 297.91it/s]


In [20]:
def prepchain( pdbfile , chain=None, savepath=None ,unid='',  verbose = False):
	assert savepath is not None
	try:
		#parse the pdb file
		parser = PDB.PDBParser()
		structure = parser.get_structure(unid, pdbfile)
		if chain:
			chainID = chain
		else:
			chainID = list(structure[0].get_chains())[0].get_id()
		chain_A = structure[0][chainID]
	except:
		print("error")
		return None
	if chain_A:
		io = PDB.PDBIO()
		io.set_structure(chain_A)
		io.save(savepath)
		fixer = PDBFixer(filename=savepath)
		fixer.findNonstandardResidues()
		fixer.replaceNonstandardResidues()
		fixer.removeHeterogens(True)
		fixer.findMissingResidues()
		fixer.findMissingAtoms()
		fixer.addMissingAtoms()
		#fixer.addMissingHydrogens(7.0)
		PDBFile.writeFile(fixer.topology, fixer.positions, open(savepath, 'w'))
		return None


In [25]:
structs = glob.glob(datadir + '*/*/*/pdbfixer_in/*.pdb')
print(f'Found {len(structs)} pdb files to process')

Found 1096 pdb files to process


In [26]:
fix_structs = True
if fix_structs:
	with ProcessPool() as pool:
		futures = [ pool.schedule( prepchain,  ( pdb , None , pdb.replace('pdbfixer_in', 'structs' ) 
										  , '' , False ) , timeout = 240) for pdb in structs  ]
		for future in tqdm.tqdm(futures, total=len(structs)):
			try:
				results = future.result()
			except TimeoutError as error:
				print("unstable_function took longer than %d seconds" % error.args[1])
			except ProcessExpired as error:
				print("%s. Exit code: %d" % (error, error.exitcode))
			except Exception as error:
				print("unstable_function raised %s" % error)
				print(error.traceback)  # Python's traceback of remote process
		pool.close()
		pool.join()


  0%|          | 0/1096 [00:00<?, ?it/s]/work/FAC/FBM/DBC/cdessim2/default/dmoi/miniconda3/envs/torch/lib/python3.12/site-packages/Bio/PDB/StructureBuilder.py:89: PDBConstructionWarning: WARNING: Chain A is discontinuous at line 1069.
  warnings.warn(
/work/FAC/FBM/DBC/cdessim2/default/dmoi/miniconda3/envs/torch/lib/python3.12/site-packages/Bio/PDB/StructureBuilder.py:89: PDBConstructionWarning: WARNING: Chain A is discontinuous at line 1170.
  warnings.warn(
/work/FAC/FBM/DBC/cdessim2/default/dmoi/miniconda3/envs/torch/lib/python3.12/site-packages/Bio/PDB/StructureBuilder.py:89: PDBConstructionWarning: WARNING: Chain B is discontinuous at line 1243.
  warnings.warn(
/work/FAC/FBM/DBC/cdessim2/default/dmoi/miniconda3/envs/torch/lib/python3.12/site-packages/Bio/PDB/StructureBuilder.py:89: PDBConstructionWarning: WARNING: Chain B is discontinuous at line 1465.
  warnings.warn(
/work/FAC/FBM/DBC/cdessim2/default/dmoi/miniconda3/envs/torch/lib/python3.12/site-packages/Bio/PDB/StructureBuil

error


/work/FAC/FBM/DBC/cdessim2/default/dmoi/miniconda3/envs/torch/lib/python3.12/site-packages/Bio/PDB/StructureBuilder.py:89: PDBConstructionWarning: WARNING: Chain A is discontinuous at line 2692.
  warnings.warn(
 10%|█         | 115/1096 [01:23<11:44,  1.39it/s]/work/FAC/FBM/DBC/cdessim2/default/dmoi/miniconda3/envs/torch/lib/python3.12/site-packages/Bio/PDB/StructureBuilder.py:89: PDBConstructionWarning: WARNING: Chain A is discontinuous at line 865.
  warnings.warn(
 12%|█▏        | 127/1096 [01:34<11:03,  1.46it/s]/work/FAC/FBM/DBC/cdessim2/default/dmoi/miniconda3/envs/torch/lib/python3.12/site-packages/Bio/PDB/StructureBuilder.py:89: PDBConstructionWarning: WARNING: Chain A is discontinuous at line 2545.
  warnings.warn(
/work/FAC/FBM/DBC/cdessim2/default/dmoi/miniconda3/envs/torch/lib/python3.12/site-packages/Bio/PDB/StructureBuilder.py:89: PDBConstructionWarning: WARNING: Chain B is discontinuous at line 2740.
  warnings.warn(
/work/FAC/FBM/DBC/cdessim2/default/dmoi/miniconda3/en

error


/work/FAC/FBM/DBC/cdessim2/default/dmoi/miniconda3/envs/torch/lib/python3.12/site-packages/Bio/PDB/PDBParser.py:388: PDBConstructionWarning: Ignoring unrecognized record 'END' at line 388
  warnings.warn(
 88%|████████▊ | 963/1096 [07:37<00:57,  2.32it/s]/work/FAC/FBM/DBC/cdessim2/default/dmoi/miniconda3/envs/torch/lib/python3.12/site-packages/Bio/PDB/PDBParser.py:388: PDBConstructionWarning: Ignoring unrecognized record 'END' at line 1577
  warnings.warn(
/work/FAC/FBM/DBC/cdessim2/default/dmoi/miniconda3/envs/torch/lib/python3.12/site-packages/Bio/PDB/PDBParser.py:388: PDBConstructionWarning: Ignoring unrecognized record 'END' at line 301
  warnings.warn(
100%|██████████| 1096/1096 [07:56<00:00,  2.30it/s]


In [30]:
#swap name to number for the pdb files and save a mapping file
#rename the pdb files in the structs directory
def rename_pdb_files(struct_dir):
	structs = glob.glob(os.path.join(struct_dir, 'structs', '*.pdb'))
	mapping = {}
	for i,pdb in enumerate(structs):
		pdb_id = os.path.basename(pdb).split('.')[0]
		new_name = f'{i}.pdb'
		new_path = os.path.join(struct_dir, 'structs', new_name)
		os.rename(pdb, new_path)
		#save the mapping
		mapping[pdb_id] = new_name
	#transform the mapping to a dataframe
	mapping_df = pd.DataFrame.from_dict(mapping, orient='index', columns=['new_name'])
	mapping_df.index.name = 'pdb_id'
	#save the mapping to a file
	mapping_df.to_csv(os.path.join(struct_dir, 'mapping.csv'))
#apply the renaming function to all struct directories
for struct_dir in tqdm.tqdm(pdb_df['struct_dir'].unique(), desc='Renaming PDB files'):
	rename_pdb_files(struct_dir)

Renaming PDB files: 100%|██████████| 70/70 [00:00<00:00, 107.18it/s]
